# NeuroAtlas — A RAG-Powered Knowledge Assistant for Mental, Neurodevelopmental, Neurological & Sleep Disorders
Domain: mental disorders, neurodevelopmental disorders, neurological/movement disorders, and
sleep disorders, sourced from WHO, NIMH, NINDS, NHLBI, NICHD, NIGMS, CDC and AASM 


text length: 117
images on page: 0


## 2.1 Load & Inspect




In [2]:
import pymupdf
def pdf_to_markdown(pdf_path, markdown_path):

    markdown = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(page.get_text())
            markdown.append("\n\n")

    # Creates/opens the output Markdown file in write mode using UTF-8 encoding.
    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown)) #one string

    print(f"Markdown file created: {markdown_path}")


pdf_to_markdown(
    "../data/documents/Neuro_dataset.pdf",
    "../data/documents/output.md"
)

Markdown file created: ../data/documents/output.md


## 2.2 Chunking Strategy



In [ ]:
import re
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Settings
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2" #speed + relatively small size.
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50 #12.5%

# Input and output files
PDF_PATH = "../data/documents/Neuro_dataset.pdf"
MARKDOWN_PATH = "../data/documents/output.md"

#Book ranges
BOOK_RANGES = [
    ("WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)", 1, 852),
    ("Fundamentals of Psychological Disorders (mental use)", 853, 1114),
    ("Handbook of Neurodevelopmental and Genetic Disorders in Adults (developmental)", 1115, 1622),
    ("Developmental Screening - CDC", 1623, 1641),
    ("Neurological Disorders - Public Health Challenges (neuro, WHO)", 1642, 1873),
    ("Global Status Report on Neurology (neuro 2, WHO)", 1874, 2158),
    ("Sleep Disorders and Sleep Deprivation (National Academies)", 2159, 2583),
    ("Rhythmic Movement Disorder case report (rmd, JCSM)", 2584, 2587),
]

# book names
def get_book_name(page_number):
    """Return the book name corresponding to a PDF page number."""

    for book_name, start_page, end_page in BOOK_RANGES:

        if start_page <= page_number <= end_page:
            return book_name

    return None 

# Tokenizer used by the embedding model
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)


# Recursive chunker
# It creates and configures the chunker that will later split your section text into ~400-token chunks with overlap,
#  while trying to split at natural boundaries first.
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [ ]:
def split_into_pages(text):
    """Split Markdown text into individual pages."""

    # Define a regular expression that matches page headings such as: ## Page 1, ## Page 25, ## Page 100
    pattern = r"^##\s*Page\s+(\d+)\s*$"

    # Split the entire Markdown text wherever a page heading is found.
    parts = re.split(
        pattern,
        text,
        flags=re.MULTILINE
    )

    # Create an empty list to store all extracted pages.
    pages = []

    # Page numbers and page contents appear alternately in the parts list.
    # Start at index 1 and move by 2: 1, 3, 5, 7, ...
    for i in range(1, len(parts), 2):

        # Get the page number and convert it from a string to an integer.
        page_number = int(parts[i])

        # Get the text belonging to this page and remove extra whitespace.
        page_text = parts[i + 1].strip()

        # Store the page number and its corresponding text in a dictionary.
        pages.append({
            "page_number": page_number,
            "text": page_text
        })

    # Return the list containing all pages and their text.
    return pages

In [ ]:
def is_heading(line):
    """Return True if a line looks like a section heading."""

    line = line.strip()

    if not line:
        return False

    # Ignore page markers
    if re.match(r"^##\s*Page\s+\d+\s*$", line):
        return False

    # Markdown heading
    if line.startswith("# "):
        return True

    # All-uppercase heading
    if line.isupper() and len(line.split()) <= 15:
        return True

    return False

In [ ]:
def split_into_sections(text):
    """Split the entire document into sections while allowing sections to continue across pages."""

    ## Convert the entire document into a list of lines.  
    lines = text.splitlines()

    sections = []
    heading = ""
    content = []
    page_start = None
    page_end = None
    current_page = None

    for line in lines:

        # Detect page marker
        page_match = re.match(r"^##\s*Page\s+(\d+)\s*$", line.strip())

        if page_match:
            current_page = int(page_match.group(1))
            continue

        # Detect a real heading
        if is_heading(line):

            # Save the previous section
            if content:
                sections.append({
                    "heading": heading,
                    "content": "\n".join(content).strip(),
                    "page_start": page_start,
                    "page_end": page_end
                })

                content = []

            heading = line.strip()
            page_start = current_page

        else:
            if line.strip():
                content.append(line)

            if current_page is not None:
                page_end = current_page

    # Save the final section
    if content:
        sections.append({
            "heading": heading,
            "content": "\n".join(content).strip(),
            "page_start": page_start,
            "page_end": page_end
        })

    return sections

In [ ]:
def chunk_document(text):
    """Create chunks from continuous sections that can span multiple pages."""

    chunks = []

    sections = split_into_sections(text)

    for section in sections:

        heading = section["heading"]
        section_text = section["content"]

        section_chunks = splitter.split_text(section_text)

        for chunk in section_chunks:

            if heading:
                chunk_text = f"{heading}\n{chunk}"
            else:
                chunk_text = chunk

            chunks.append({
                "text": chunk_text,
                "book_name": get_book_name(section["page_start"]),
                "section": heading,
                "page_start": section["page_start"],
                "page_end": section["page_end"]
            })

    return chunks

In [ ]:
import json

with open("../data/documents/chunks.json", "w", encoding="utf-8") as file:
    json.dump(chunks, file, ensure_ascii=False, indent=2)

In [ ]:
print("First chunk:\n")
print(chunks[0]["text"])
print("\nMetadata:")
print("Book:", chunks[0]["book_name"])
print("Pages:", chunks[0]["page_start"], "-", chunks[0]["page_end"])

print("Section:", chunks[0]["section"])

FileNotFoundError: no such file: '../data/documents/Neuro_dataset_repaired.pdf'

## 2.3 Embeddings & Vector Store